# kmedoids 3D Demo

This small notebook exercises the newly added `find_optimal_k_points_kmedoids_3D` on synthetic 3D data.

It compares it against GBM 3D for a quick sanity check.

In [ ]:
import numpy as np
import pandas as pd

print("Creating tiny synthetic 3D volume...")

In [ ]:
# Very small 3D synthetic field for fast demo
lw = 2
W, H_img, D = 100 * lw, 100, 8
x = np.linspace(-20, 20, W)
y = np.linspace(0.4, 12, D)
z = np.linspace(-70, 70, H_img)

X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
C = 5e-6 + 3e-6 * np.exp(-((X)**2 + (Z)**2 + (Y-5)**2) / 60)

df = pd.DataFrame({
    "X": X.ravel(),
    "Y": Y.ravel(),
    "Z": Z.ravel(),
    "Carbon": C.ravel()
})

inside = np.ones(W * H_img * D, dtype=bool)
global_mean = float(C.mean())

print(f"Volume: {W}x{H_img}x{D}, mean ≈ {global_mean:.2e}")

In [ ]:
# Try importing and running both methods
k = 5

print("\nRunning GBM 3D...")
try:
    from gbm.core import find_optimal_k_points_gbm_3D
    loss_gbm, mean_sens, std_sens, pos_gbm = find_optimal_k_points_gbm_3D(
        df, inside, k=k, in_CO2_avg=global_mean,
        cross_section="XZ", barn_LW_ratio=lw,
        epochs=4, sampling_budget=300
    )
    print(f"GBM 3D loss: {loss_gbm:.3e}")
except Exception as e:
    print(f"GBM 3D failed: {e}")

In [ ]:
print("\nRunning kmedoids 3D...")
try:
    from gbm.baselines.kmedoids import find_optimal_k_points_kmedoids_3D
    loss_kmd, mean_sens, std_sens, pos_kmd = find_optimal_k_points_kmedoids_3D(
        df, inside, k=k, in_CO2_avg=global_mean,
        barn_LW_ratio=lw, epochs=4, sampling_budget=300
    )
    print(f"kmedoids 3D loss: {loss_kmd:.3e}")
except Exception as e:
    print(f"kmedoids 3D failed: {e}")
    print("(Make sure you have appended the kmedoids_3D function to the file)")

**Notes**

- This is a minimal demo on synthetic data.
- kmedoids 3D uses downsampling + PAM + subgradient refinement.
- For real results, use the full experiment scripts in `experiments/`.